# Piloto Text2KG — Qwen2.5-7B-Instruct-AWQ

**Configuração obrigatória antes de salvar e rodar:**
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Settings → Secrets → adicione `GITHUB_TOKEN` e `HF_TOKEN` (**marque "Attach to notebook"** para cada um)
4. Para retomar: Add Data → Your Work → versão anterior → **Add as input**
5. Para rodar: Save Version → **Save & Run All** (não Run All Cells)

**Tempo estimado:** ~3h (200 frases × 2 ontologias × 3 condições + indução C2)

In [ ]:
# Célula 1 — Configuração
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN     = secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

REPO_SLUG  = "fmsoares/does-ontology-matter"
REPO_URL   = f"https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git"
BENCH_URL  = "https://github.com/nandana/Text2KGBench.git"

WORK_DIR   = "/kaggle/working"
REPO_DIR   = f"{WORK_DIR}/does-ontology-matter"
BENCH_DIR  = f"{WORK_DIR}/Text2KGBench"
RESULT_DIR = f"{REPO_DIR}/resultados/piloto"
C2_DIR     = f"{REPO_DIR}/configs/c2"
VLLM_LOG   = f"{WORK_DIR}/vllm.log"
MODELO     = "Qwen/Qwen2.5-7B-Instruct-AWQ"
PORT       = 8000

GIT_EMAIL  = "felipemoraissoares@gmail.com"
GIT_NAME   = "Felipe Morais (Kaggle)"

print(f"REPO_DIR:   {REPO_DIR}")
print(f"BENCH_DIR:  {BENCH_DIR}")
print(f"RESULT_DIR: {RESULT_DIR}")

In [ ]:
# Célula 2 — Helpers
import subprocess

def run(cmd, check=True, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Falhou (rc={result.returncode}): {cmd[:120]}")
    return result

def git(cmd):
    return run(f'git -C "{REPO_DIR}" {cmd}', check=False)

In [ ]:
# Célula 3 — Clonar repositórios
if not os.path.exists(REPO_DIR):
    run(f"git clone --depth=1 {REPO_URL} {REPO_DIR}")
else:
    print("Repo já existe — atualizando...")
    git("pull")

if not os.path.exists(BENCH_DIR):
    run(f"git clone --depth=1 {BENCH_URL} {BENCH_DIR}")
else:
    print("Text2KGBench já existe.")

git(f"config user.email '{GIT_EMAIL}'")
git(f"config user.name '{GIT_NAME}'")
print("Clones OK.")

In [ ]:
# Célula 4 — Restaurar resultados anteriores (sessão interrompida)
# Se uma versão anterior foi anexada como input (Add Data → Your Work),
# ela aparece em /kaggle/input/{dataset-slug}/.
# Copiamos os arquivos para RESULT_DIR para que --skip_existentes os pule.

import glob, shutil

os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(C2_DIR, exist_ok=True)

restaurados = 0
input_base = "/kaggle/input"
if os.path.exists(input_base):
    for dataset_name in os.listdir(input_base):
        # Procura JSONLs de resultados em qualquer subdirectório
        for src in glob.glob(f"{input_base}/{dataset_name}/**/*.jsonl", recursive=True):
            dest = os.path.join(RESULT_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest)
                restaurados += 1
                print(f"  Restaurado: {os.path.basename(src)}")
        # Restaura curvas de calibracao C2
        for src in glob.glob(f"{input_base}/{dataset_name}/**/calibracao_*.json", recursive=True):
            dest = os.path.join(C2_DIR, os.path.basename(src))
            if not os.path.exists(dest):
                shutil.copy2(src, dest)
                restaurados += 1
                print(f"  Restaurado C2: {os.path.basename(src)}")

if restaurados > 0:
    print(f"\nTotal restaurado: {restaurados} arquivo(s) — experimentos existentes serao pulados.")
else:
    print("Nenhum resultado anterior encontrado — execucao completa.")

In [ ]:
# Célula 5 — Instalar dependências
run("pip install -q vllm openai sentence-transformers scikit-learn nltk scipy numpy pandas statsmodels")

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import sys
sys.path.insert(0, f"{REPO_DIR}/src")
print("Dependências OK.")

In [ ]:
# Célula 6 — Subir vLLM em segundo plano
# dtype=half obrigatório para T4 (fp16). tensor-parallel-size=2 usa ambas as GPUs T4.

vllm_cmd = (
    f"vllm serve {MODELO} "
    f"--port {PORT} "
    f"--dtype half "
    f"--tensor-parallel-size 2 "
    f"--max-model-len 4096 "
    f"--gpu-memory-utilization 0.90 "
    f"> {VLLM_LOG} 2>&1 &"
)
os.system(vllm_cmd)
print(f"vLLM iniciado em segundo plano. Log: {VLLM_LOG}")

In [ ]:
# Célula 7 — Aguardar vLLM pronto (polling automático /health)
import time, urllib.request, urllib.error

VLLM_URL = f"http://localhost:{PORT}"
TIMEOUT  = 360  # segundos
INTERVALO = 10
elapsed   = 0

while elapsed < TIMEOUT:
    try:
        urllib.request.urlopen(f"{VLLM_URL}/health", timeout=3)
        print(f"vLLM pronto após {elapsed}s.")
        break
    except (urllib.error.URLError, ConnectionRefusedError, OSError):
        if elapsed % 30 == 0:
            print(f"  aguardando vLLM... ({elapsed}s / {TIMEOUT}s)")
        time.sleep(INTERVALO)
        elapsed += INTERVALO
else:
    print(f"ERRO: vLLM não respondeu em {TIMEOUT}s. Últimas linhas do log:")
    run(f"tail -40 {VLLM_LOG}", check=False)
    raise RuntimeError("vLLM não iniciou. Verifique GPU e modelo.")

In [ ]:
# Célula 8 — Gerar manifesto do piloto com caminhos do Kaggle
import json

run(
    f"python {REPO_DIR}/scripts/gerar_manifesto.py "
    f"--url http://localhost:{PORT}/v1 "
    f"--saida_dir {REPO_DIR}/configs/pipeline",
    cwd=REPO_DIR
)

piloto_src = f"{REPO_DIR}/configs/pipeline/manifesto_piloto_kaggle.json"
with open(piloto_src, encoding="utf-8") as f:
    piloto = json.load(f)

for exp in piloto:
    exp["saida"] = f"{RESULT_DIR}/{os.path.basename(exp['saida'])}"

piloto_path = f"{WORK_DIR}/manifesto_piloto_kaggle.json"
with open(piloto_path, "w", encoding="utf-8") as f:
    json.dump(piloto, f, ensure_ascii=False, indent=2)

print(f"Manifesto com {len(piloto)} experimentos -> {piloto_path}")
for exp in piloto:
    existe = "[ja existe]" if os.path.exists(exp['saida']) else "[pendente]"
    print(f"  {exp['id']}  {existe}")

In [ ]:
# Célula 9 — Executar piloto com push automático após cada experimento
# --skip_existentes: pula arquivos já existentes (retomada automática)
# --auto_push: commit + push ao GitHub após cada experimento

PYPATH = {**os.environ, "PYTHONPATH": f"{REPO_DIR}/src"}

run(
    f"python -m t2kg.run "
    f"--manifesto {piloto_path} "
    f"--bench_dir {BENCH_DIR} "
    f"--skip_existentes "
    f"--auto_push "
    f"--push_repo_dir \"{REPO_DIR}\" "
    f"--verbose",
    cwd=REPO_DIR,
    env=PYPATH
)
print("\nPiloto concluído.")

In [ ]:
# Célula 10 — Indução C2: extração livre nas 50 frases de treino (D-16)
# Produz ontologia induzida E curvas de calibração (agrupamento + projeção, 0.70-0.90).
# Roda com o mesmo vLLM já carregado.

for onto in ["1_movie", "2_music"]:
    calib_file = f"{C2_DIR}/calibracao_tekgen_{onto}.json"
    if os.path.exists(calib_file):
        print(f"\n--- tekgen/{onto}: calibracao ja existe, pulando ---")
        continue
    print(f"\n--- Induzindo C2 para tekgen/{onto} ---")
    run(
        f"python {REPO_DIR}/scripts/induzir_c2.py "
        f"--dataset tekgen "
        f"--onto {onto} "
        f"--url http://localhost:{PORT}/v1 "
        f"--modelo_llm {MODELO} "
        f"--threshold 0.80 "
        f"--n_frases 50 "
        f"--seed 42 "
        f"--calibrar "
        f"--saida_dir {C2_DIR}",
        cwd=REPO_DIR,
        env=PYPATH
    )

print("\nIndução C2 concluída.")

# Push final dos arquivos C2
git("add -A")
git('commit -m "auto: C2 inducao e curvas de calibracao (D-16)"')
git("push")
print("Push C2 OK.")

In [ ]:
# Célula 11 — Avaliar resultados e gerar tabela CSV

CSV_OUT = f"{RESULT_DIR}/tabela_piloto.csv"

run(
    f"python {REPO_DIR}/scripts/avaliar_resultados.py "
    f"--manifesto {piloto_path} "
    f"--bench_dir {BENCH_DIR} "
    f"--csv {CSV_OUT}",
    cwd=REPO_DIR,
    env=PYPATH
)

import pandas as pd
df = pd.read_csv(CSV_OUT)
print(df.to_string())

# Push final da tabela
git("add -A")
git('commit -m "auto: tabela_piloto.csv"')
git("push")
print("Push tabela OK.")

In [ ]:
# Célula 12 — Verificar log do vLLM (diagnóstico)
run(f"tail -30 {VLLM_LOG}", check=False)